==============================================================================
LlamaParse 잔여 실패 페이지 재처리 — 독립 실행 노트북

메인 파이프라인(build_rag_index_v5_2_pymupdf_only.ipynb)을 2단계(pdfplumber+
PyMuPDF)까지만 돌리고 100개사 본작업을 끝낸 뒤, 그중 캐스케이딩 후에도
실패가 남은 페이지만 여기서 LlamaParse로 재처리해서 기존 Chroma DB에
추가 적재함.

전제조건:
  1. 메인 파이프라인의 [Cell 7c]가 만든 extraction_diagnostics.csv가 있어야 함
     (still_failed_pages 컬럼 포함된 v5.2 이상 버전)
  2. 메인 파이프라인으로 이미 Chroma DB가 구축돼 있어야 함 (Drive 백업 경로 동일)

속도 최적화: 페이지 하나당 API 호출 1번이 아니라, 문서(doc_id) 하나당
target_pages에 실패 페이지 전체를 한꺼번에 넣어서 API 호출 1번으로 처리함
(LlamaParse는 target_pages에 콤마로 여러 페이지를 받을 수 있음).

비용 관리: SELECTION_RULE로 재처리 대상을 좁힐 수 있음 (기본: 잔여 실패
페이지가 1개라도 있으면 전부 대상 — 10개사 실측 기준 전체 비용이 작아서
굳이 더 좁힐 필요는 없었지만, 필요하면 MIN_ISSUE_RATIO를 올릴 것).
==============================================================================

### [Cell 1] 설치 + Drive 마운트 + API 키

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma langchain-community \
    chromadb pandas llama-cloud-services

from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API Key 입력 (임베딩용): ")
os.environ["LLAMA_CLOUD_API_KEY"] = getpass.getpass("LlamaParse API Key 입력: ")

### [Cell 2] 설정값 — 메인 파이프라인의 CONFIG와 경로를 맞출 것

In [ ]:
CONFIG = {
    "embedding_model": "gemini-embedding-001",
    "chunk_size_tokens": 800,
    "chunk_overlap_tokens": 100,

    # 메인 파이프라인과 동일하게 맞출 것 (로컬 작업 + Drive는 백업 전용)
    "persist_dir": "/content/chroma_db_local",
    "drive_backup_dir": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/chroma_db",

    "extraction_diag_csv": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과/extraction_diagnostics.csv",

    # LlamaParse
    "llamaparse_mode": "cost_effective",  # 페이지당 3크레딧(≈$0.00375). 필요시 "fast"(1크레딧)
    "llamaparse_language": "ko",

    # 재처리 대상 선정 규칙 — 기본은 "잔여 실패 페이지가 1개라도 있으면 전부".
    # 예산이 걱정되면 0.3 등으로 올려서 심한 문서만 우선 처리 가능.
    "min_issue_ratio_after": 0.0,
}

### [Cell 3] 라이브러리 import + 로컬 Chroma 복원

In [ ]:
import ast
import json
import shutil

import pandas as pd
import tiktoken
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

try:
    _enc = tiktoken.encoding_for_model(CONFIG["embedding_model"])
except KeyError:
    _enc = tiktoken.get_encoding("cl100k_base")


def token_len(text: str) -> int:
    return len(_enc.encode(text))


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size_tokens"],
    chunk_overlap=CONFIG["chunk_overlap_tokens"],
    length_function=token_len,
    separators=["\n\n", "\n", ". ", " ", ""],
)


# 메인 파이프라인과 동일한 복원 로직 (Drive에 직접 Chroma를 물리면 손상 위험 있음 — 실제로 겪은 사고)
def restore_local_from_drive_backup():
    if os.path.exists(CONFIG["persist_dir"]):
        print("로컬 인덱스 이미 존재 (복원 스킵)")
        return
    if os.path.exists(CONFIG["drive_backup_dir"]):
        print("Drive 백업 -> 로컬 복원 중")
        shutil.copytree(CONFIG["drive_backup_dir"], CONFIG["persist_dir"])
        print("복원 완료")
    else:
        raise FileNotFoundError(
            "Drive 백업이 없음 — 메인 파이프라인을 먼저 돌려서 Chroma DB를 만들어야 함"
        )


def backup_local_to_drive():
    shutil.copytree(CONFIG["persist_dir"], CONFIG["drive_backup_dir"], dirs_exist_ok=True)


restore_local_from_drive_backup()
embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
vectorstore = Chroma(persist_directory=CONFIG["persist_dir"], embedding_function=embeddings)
print("기존 Chroma DB 로드 완료")

### [Cell 4] 재처리 대상 선정 — extraction_diagnostics.csv에서 잔여 실패 문서만 추출

In [ ]:
diag_df = pd.read_csv(CONFIG["extraction_diag_csv"])

targets = diag_df[
    (diag_df["issue_pages_after"] > 0)
    & (diag_df["issue_ratio_after"] >= CONFIG["min_issue_ratio_after"])
].copy()

# still_failed_pages는 CSV에 문자열(JSON 리스트)로 저장돼 있으므로 파싱
targets["still_failed_pages"] = targets["still_failed_pages"].apply(json.loads)

total_pages = targets["still_failed_pages"].apply(len).sum()
print(f"재처리 대상: {len(targets)}개 문서, 총 {total_pages}페이지")
print(targets[["company_code", "doc_type", "issue_pages_after", "issue_ratio_after"]].to_string(index=False))

# 실제 API 호출 전 예상 크레딧/비용 확인
_CREDITS_PER_PAGE = 3 if CONFIG["llamaparse_mode"] == "cost_effective" else 1
est_credits = total_pages * _CREDITS_PER_PAGE
print(f"\n예상 크레딧: {est_credits} (약 ${est_credits * 0.00125:.2f}) "
      f"— 무료 월 10,000크레딧 안에 들어오는지 확인할 것")

### [Cell 5] LlamaParse 재처리 함수 — 문서당 API 호출 1번 (모든 실패 페이지 한꺼번에)

In [ ]:
_LLAMAPARSE_MODE_MAP = {
    "fast": "parse_page_without_llm",
    "cost_effective": "parse_page_with_llm",
}


def llamaparse_rescue_pages(file_path: str, page_numbers_1indexed: list[int]) -> dict[int, str]:
    """실패 페이지 전체를 한 번의 API 호출로 재처리.
    반환: {1-indexed 페이지 번호: 마크다운 텍스트}

    주의: LlamaParse의 target_pages는 0-indexed이므로 변환 필요.
    결과 순서는 요청한 페이지 오름차순으로 반환된다고 가정 — 처음 실행 시
    페이지 번호가 실제로 맞는지 반드시 원본 PDF와 대조해서 확인할 것.
    """
    from llama_cloud_services import LlamaParse  # noqa: 지연 import

    sorted_pages = sorted(page_numbers_1indexed)
    zero_indexed = [p - 1 for p in sorted_pages]
    target_pages_str = ",".join(str(p) for p in zero_indexed)

    parser = LlamaParse(
        api_key=os.environ.get("LLAMA_CLOUD_API_KEY"),
        result_type="markdown",
        parse_mode=_LLAMAPARSE_MODE_MAP.get(CONFIG["llamaparse_mode"], "parse_page_with_llm"),
        language=CONFIG["llamaparse_language"],
        target_pages=target_pages_str,
    )
    result = parser.parse(file_path)
    docs = result.get_markdown_documents(split_by_page=True)

    if len(docs) != len(sorted_pages):
        print(f"  ⚠️ 요청 페이지 수({len(sorted_pages)})와 반환 문서 수({len(docs)})가 다름 "
              f"— 페이지 매칭이 틀렸을 수 있음. 아래 결과를 직접 확인할 것.")

    return {page: doc.text for page, doc in zip(sorted_pages, docs)}

### [Cell 6] (권장) 1개 문서로 먼저 소규모 테스트
본격 루프 전에, 페이지 매칭이 실제로 맞는지 원본 PDF와 대조해서 확인할 것.

In [ ]:
_test_row = targets.iloc[0]
_test_pages = _test_row["still_failed_pages"][:2]  # 처음 2페이지만
print(f"테스트: {_test_row['doc_id']}, 페이지 {_test_pages}")

_test_result = llamaparse_rescue_pages(_test_row["file_path"], _test_pages)
for page, text in _test_result.items():
    print(f"\n--- p.{page} (앞 300자) ---")
    print(text[:300])

print("\n원본 PDF를 열어서 위 페이지 번호·내용이 실제로 일치하는지 확인한 뒤 다음 셀로 진행할 것.")

### [Cell 7] 본 실행 — 전체 재처리 + Chroma에 추가 적재

In [ ]:
rescue_log = []

for _, row in targets.iterrows():
    doc_id = row["doc_id"]
    file_path = row["file_path"]
    pages = row["still_failed_pages"]
    print(f"처리 중: {doc_id} ({len(pages)}페이지)")

    try:
        rescued = llamaparse_rescue_pages(file_path, pages)
    except Exception as e:
        print(f"  실패 (건너뜀): {e}")
        rescue_log.append({"doc_id": doc_id, "status": "api_error", "detail": str(e)})
        continue

    new_chunks = []
    for page, text in rescued.items():
        if not text.strip():
            continue
        for i, chunk_text in enumerate(splitter.split_text(text)):
            if len(chunk_text.strip()) < 20:
                continue
            # [주의] 메인 파이프라인의 순차 counter와 절대 겹치지 않도록
            # "llama" 프리픽스로 별도 네임스페이스 사용
            chunk_id = f"{doc_id}_llama_p{page:04d}_{i:02d}"
            meta = {
                "company_code": row["company_code"],
                "company_name": row["company_name"],
                "year": int(row["year"]),
                "doc_type": row["doc_type"],
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "page": page,
                "section_title": "",  # LlamaParse 결과는 섹션 매핑 안 함
                "content_type": "text",  # LlamaParse 마크다운 표는 텍스트 안에 인라인으로 포함됨
                "table_data": "",
                "esg_area": "",
                "applicable_item_codes": "[]",
                "embedding_model": CONFIG["embedding_model"],
                "indexed_at": pd.Timestamp.now().isoformat(),
                "source_file_hash": "llamaparse_rescue",
                "extraction_source": "llamaparse",  # [신규] 이 청크가 어느 단계에서 왔는지 추적용
            }
            meta["chunk_char_count"] = len(chunk_text)
            new_chunks.append(Document(page_content=chunk_text, metadata=meta))

    if new_chunks:
        vectorstore.add_documents(new_chunks, ids=[c.metadata["chunk_id"] for c in new_chunks])
        print(f"  -> {len(new_chunks)}개 청크 추가 적재")

    rescue_log.append({
        "doc_id": doc_id, "status": "ok",
        "pages_requested": len(pages), "pages_returned": len(rescued),
        "chunks_added": len(new_chunks),
    })

print(f"\n완료: {len(rescue_log)}건 처리")
print("최종 Drive 백업 중...")
backup_local_to_drive()
print("백업 완료")

### [Cell 8] 결과 로그 저장 + 요약

In [ ]:
log_df = pd.DataFrame(rescue_log)
log_path = CONFIG["extraction_diag_csv"].replace(
    "extraction_diagnostics.csv", "llamaparse_rescue_log.csv"
)
log_df.to_csv(log_path, index=False, encoding="utf-8-sig")
print(f"로그 저장: {log_path}")
print(log_df.to_string(index=False))

if "status" in log_df.columns:
    n_ok = (log_df["status"] == "ok").sum()
    n_err = (log_df["status"] == "api_error").sum()
    print(f"\n성공: {n_ok}건 / 실패: {n_err}건")
    if n_err > 0:
        print("실패 건은 이 노트북을 다시 실행하면 됨 — Chroma add는 chunk_id 기반이라 "
              "성공한 문서를 다시 돌려도 중복 적재되지 않음 (동일 ID면 덮어씀).")